In [14]:

#Cargamos librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.special as special
from scipy.optimize import curve_fit
import seaborn as sns
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

In [15]:
#Carga desde un archivo .csv sin indice
df = pd.read_csv('Base_Paris_LIMPIA.csv')
df.head(5)

,Unnamed: 0,id,scrape_id,host_id,latitude,longitude,listing_url,last_scraped,source,name,...,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,reviews_per_month
0,0,3109,20240906025355,3631,48.831910,2.318700,https://www.airbnb.com/rooms/3109,2024-09-11,city scrape,zen and calm,...,5.00,5.00,5.00,5.00,5.00,5.00,5.00,1.0,1.0,0.05
1,1,5396,20240906025355,7903,48.852470,2.358350,https://www.airbnb.com/rooms/5396,2024-09-13,city scrape,Your perfect Paris studio on Île Saint-Louis,...,4.61,4.64,4.59,4.81,4.84,4.96,4.59,1.0,1.0,2.23
2,2,7397,20240906025355,2626,48.859090,2.353150,https://www.airbnb.com/rooms/7397,2024-09-06,city scrape,MARAIS - 2ROOMS APT - 2/4 PEOPLE,...,4.73,4.81,4.45,4.92,4.89,4.93,4.74,1.0,1.0,2.20
3,3,7964,20240906025355,22155,48.874170,2.342450,https://www.airbnb.com/rooms/7964,2024-09-10,previous scrape,Sunny apartment with balcony,...,4.80,5.00,5.00,5.00,5.00,5.00,5.00,1.0,1.0,0.03
4,4,241715,20240906025355,3342097,48.893464,2.378341,https://www.airbnb.com/rooms/241715,2024-09-11,city scrape,Big Cosy Appartement with 100 m2 Terrace in Paris,...,4.80,4.80,4.70,4.90,4.90,4.90,4.70,1.0,1.0,0.80


In [16]:
#Sustituir valores nulos por un string en  concreto 
df["host_response_rate"] =df["host_response_rate"].replace('No captured',0) 
# Elimina cualquier símbolo no numérico (por ejemplo, '%') y convierte a tipo float
df['host_response_rate'] = df['host_response_rate'].str.replace('%', '').astype(float)

#Sustituir valores nulos por un string en  concreto 
df["host_acceptance_rate"] =df["host_acceptance_rate"].replace('No captured',0) 
# Elimina cualquier símbolo no numérico (por ejemplo, '%') y convierte a tipo float
df['host_acceptance_rate'] = df['host_acceptance_rate'].str.replace('%', '').astype(float)


In [17]:
#Rellenamos nulos
df =df.fillna(method="bfill")
df =df.fillna(method="ffill")

C:\Users\sarah\AppData\Local\Temp\ipykernel_26548\1655763348.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df =df.fillna(method="bfill")
C:\Users\sarah\AppData\Local\Temp\ipykernel_26548\1655763348.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df =df.fillna(method="ffill")


In [18]:
nulos=df.isnull().sum().sum()
nulos

0

In [19]:
#ajustar max de filas
pd.options.display.max_rows=10
cuantitativas=df.iloc[ : , 40:71]
cuantitativas

,host_listings_count,host_total_listings_count,accommodates,bathrooms,bedrooms,beds,price,minimum_nights,maximum_nights,minimum_minimum_nights,...,number_of_reviews_l30d,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes
0,1.0,1.0,2.0,1.0,1.0,1.0,113.0,2.0,30.0,2.0,...,0.0,5.00,5.00,5.00,5.00,5.00,5.00,5.00,1.0,1.0
1,2.0,4.0,2.0,1.0,0.0,1.0,95.0,2.0,730.0,2.0,...,0.0,4.61,4.64,4.59,4.81,4.84,4.96,4.59,1.0,1.0
2,1.0,10.0,4.0,1.0,2.0,1.0,145.0,2.6,130.0,2.0,...,0.0,4.73,4.81,4.45,4.92,4.89,4.93,4.74,1.0,1.0
3,1.0,1.0,3.0,1.0,2.0,1.5,171.9,7.0,365.0,7.0,...,0.0,4.80,5.00,5.00,5.00,5.00,5.00,5.00,1.0,1.0
4,1.0,2.0,6.0,1.0,3.0,0.0,450.0,5.0,120.0,5.0,...,0.0,4.80,4.80,4.70,4.90,4.90,4.90,4.70,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95456,2.0,3.0,4.0,1.0,1.0,2.0,77.0,1.0,365.0,1.0,...,0.0,5.00,5.00,5.00,5.00,5.00,5.00,5.00,2.0,2.0
95457,1.4,7.0,6.0,1.0,3.0,3.0,171.9,2.0,365.0,2.0,...,0.0,4.80,4.80,4.70,4.90,4.90,4.90,4.70,1.2,1.0
95458,1.4,2.0,1.0,1.0,1.3,1.0,38.0,1.0,365.0,1.0,...,0.0,4.80,4.80,4.70,4.90,4.90,4.90,4.70,1.2,0.0
95459,1.4,2.0,1.0,1.0,1.3,2.0,43.0,2.6,1125.0,2.5,...,0.0,4.80,4.80,4.70,4.90,4.90,4.90,4.70,1.2,1.0


In [ ]:
#Obtenemos el limite superior y el límite inferior de la columna objetivo
Max=df['price'].max()
Min=df['price'].min()
Limites= [Min, Max]
Limites

In [ ]:
#Categorización de variables
#Declaramos 2 intervalos 
intervalos=np.linspace(104.0, 2981.0, 3)
intervalos

In [ ]:
#Creamos las categorías 
categorias= ["Min.Price", "Max.Price"]

In [ ]:
#Finalmente creamos las categorías en la columna numérica
df['price']=pd.cut(x= df['price'], bins=intervalos, labels= categorias )

df['price']

In [ ]:
df['price'].isnull().sum()

In [24]:
#Declaramos las variables dependientes e independientes para la regresión Logística
Vars_Indep= df[['bathrooms', 'bedrooms', 'beds']]
Var_Dep= df['price']

#Redefinimos las variables 
X= Vars_Indep
y= Var_Dep

In [25]:
#Dividimos el conjunto de datos en la parte de entrenamiento y prueba:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state =None)

In [26]:

#Se escalan todos los datos
escalar = StandardScaler()

In [27]:
#Para realizar el escalamiento de las variables “X” tanto de entrenamiento como de prueba, utilizaremos fit_transform
X_train = escalar.fit_transform(X_train)
X_test = escalar.transform(X_test)

In [28]:
#Definimos el algoritmo a utilizar
from sklearn.linear_model import LogisticRegression
algoritmo= LogisticRegression()

In [29]:
#Entrenamos el modelo
algoritmo.fit(X_train, y_train)

ValueError: Unknown label type: continuous. Maybe you are trying to fit a classifier, which expects discrete classes on a regression target with continuous values.